## Spike-and-Slab Bayesian Linear Regression (Pyro)

This model is a **Bayesian linear regression** with a **spike-and-slab prior** on the regression coefficients.  
It is useful when we expect only a small subset of features to be relevant (i.e., a **sparse** model).

### Model definition

For each coefficient $\beta_j$, we introduce:

- a binary **inclusion indicator** $\gamma_j \in \{0,1\}$
- a continuous latent variable $z_j$

The spike-and-slab prior is:

$$
\gamma_j \sim \text{Bernoulli}(1-\pi)
$$

$$
z_j \sim \mathcal{N}(0, \tau^2)
$$

$$
\beta_j = \gamma_j z_j \sim (1-\pi)\mathcal{N}(0, \tau^2)+\pi\delta_0
$$

where:

- $\pi$ is the **spike probability** (probability that the coefficient is exactly zero),
- $\tau$ is the slab standard deviation.
- $\delta_0$ is the dirac measure at zero.

This construction gives:

- $\beta_j = 0$ exactly when $\gamma_j = 0$ (**spike at zero**),
- $\beta_j \sim \mathcal{N}(0, \tau^2)$ when $\gamma_j = 1$ (**slab**).

### Likelihood

Given predictors $X \in \mathbb{R}^{n \times p}$, the regression model is:

$$
y_i \sim \mathcal{N}\left(\alpha + x_i^\top \beta,\; \sigma^2\right)
$$

where:

- $\alpha$ is the intercept,
- $\sigma$ is the observation noise standard deviation,
- $\delta_0$ is the dirac measure at zero.

In the Pyro implementation, we also place priors on:

- **Intercept**: $\alpha \sim \mathcal{N}(0, 5^2)$
- **Noise scale**: $\sigma \sim \text{LogNormal}(0, 0.5)$

---

## Variational Inference (Spike-and-Slab Guide)

To approximate the posterior, we use a **mean-field variational guide** with the same spike-and-slab structure:

$$
q(\gamma_j) = \text{Bernoulli}(\alpha_j)
$$

$$
q(z_j) = \mathcal{N}(\mu_j, s_j^2)
$$

So the variational posterior for each coefficient is:

$$
\beta_j = \gamma_j z_j  \sim \alpha_j\mathcal{N}(\mu_j, s_j^2)+(1-\alpha_j)\delta_0
$$

where the variational parameters are:

- $\alpha_j$: posterior inclusion probability
- $\mu_j$: slab mean
- $s_j$: slab standard deviation

We also use variational distributions for the intercept and noise:

$$
q(\alpha) = \mathcal{N}(m_\alpha, s_\alpha^2)
$$

$$
q(\sigma) = \text{LogNormal}(m_\sigma, s_\sigma^2)
$$

---

## Interpretation of variational parameters

After training:

- `sigmoid(beta_alpha_logit)` gives the **posterior inclusion probabilities** $\alpha_j$
- `beta_mu` gives the slab means $\mu_j$

A useful summary of the posterior mean coefficient is:

$$
\mathbb{E}_q[\beta_j] = \mathbb{E}_q[\gamma_j z_j]
\approx \mathbb{E}_q[\gamma_j] \; \mathbb{E}_q[z_j]
= \alpha_j \mu_j
$$

So in code, a common estimator is:

```python
alpha = torch.sigmoid(pyro.param("beta_alpha_logit"))
mu = pyro.param("beta_mu")
beta_post_mean = alpha * mu

In [1]:
import torch
import pyro
import pyro.distributions as dist
import pyro.distributions.constraints as constraints
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.optim import Adam

pyro.clear_param_store()
torch.manual_seed(0)

# --------------------------------------------------
# Spike-and-slab Bayesian linear regression in Pyro
# Prior:
#   gamma_j ~ Bernoulli(1 - pi)     (gamma_j=0 => spike at zero)
#   z_j     ~ Normal(0, slab_scale)
#   beta_j  = gamma_j * z_j
#
# Variational guide:
#   q(gamma_j) = Bernoulli(alpha_j)
#   q(z_j)     = Normal(mu_j, sigma_j)
# --------------------------------------------------

class SpikeSlabLinearRegression:
    def __init__(self, n_features, pi=0.8, slab_scale=1.0, device="cpu"):
        self.p = n_features
        self.pi = pi
        self.slab_scale = slab_scale
        self.device = device

    def model(self, X, y=None):
        X = X.to(self.device)
        if y is not None:
            y = y.to(self.device)

        p = X.shape[1]

        # Spike-and-slab prior for coefficients
        gamma = pyro.sample(
            "beta_gamma",
            dist.Bernoulli(
                probs=torch.full((p,), 1.0 - self.pi, device=self.device)
            ).to_event(1)
        )  # shape [p], entries in {0,1}

        z = pyro.sample(
            "beta_z",
            dist.Normal(
                torch.zeros(p, device=self.device),
                torch.full((p,), self.slab_scale, device=self.device)
            ).to_event(1)
        )  # shape [p]

        beta = gamma * z  # exact zeros when gamma=0

        # Prior for intercept
        intercept = pyro.sample("intercept", dist.Normal(
            torch.tensor(0.0, device=self.device),
            torch.tensor(5.0, device=self.device)
        ))

        # Noise std prior
        sigma = pyro.sample("sigma", dist.LogNormal(
            torch.tensor(0.0, device=self.device),
            torch.tensor(0.5, device=self.device)
        ))

        mean = intercept + X.matmul(beta)

        with pyro.plate("data", X.shape[0]):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

    def guide(self, X, y=None):
        X = X.to(self.device)
        p = X.shape[1]

        # q(gamma): Bernoulli(alpha)
        alpha_logit = pyro.param(
            "beta_alpha_logit",
            -1.5 * torch.ones(p, device=self.device)   # init incl prob ~ 0.18
        )
        alpha = torch.sigmoid(alpha_logit)

        pyro.sample(
            "beta_gamma",
            dist.Bernoulli(probs=alpha).to_event(1)
        )

        # q(z): Normal(mu, sigma)
        beta_mu = pyro.param(
            "beta_mu",
            0.01 * torch.randn(p, device=self.device)
        )
        beta_rho = pyro.param(
            "beta_rho",
            -3.0 * torch.ones(p, device=self.device)
        )
        beta_sigma = torch.nn.functional.softplus(beta_rho) + 1e-6

        pyro.sample(
            "beta_z",
            dist.Normal(beta_mu, beta_sigma).to_event(1)
        )

        # q(intercept)
        intercept_loc = pyro.param(
            "intercept_loc", torch.tensor(0.0, device=self.device)
        )
        intercept_scale = pyro.param(
            "intercept_scale", torch.tensor(0.5, device=self.device),
            constraint=constraints.positive
        )
        pyro.sample("intercept", dist.Normal(intercept_loc, intercept_scale))

        # q(sigma)
        sigma_loc = pyro.param(
            "sigma_loc", torch.tensor(0.0, device=self.device)
        )
        sigma_scale = pyro.param(
            "sigma_scale", torch.tensor(0.2, device=self.device),
            constraint=constraints.positive
        )
        pyro.sample("sigma", dist.LogNormal(sigma_loc, sigma_scale))


# ---------------------------
# Example usage
# ---------------------------
# Simulated sparse linear data
n, p = 1000, 20
X = torch.randn(n, p)

beta_true = torch.zeros(p)
beta_true[0] = 2.5
beta_true[1] = -1.8
beta_true[2] = 1.2
intercept_true = 0.7
sigma_true = 0.1

y = intercept_true + X @ beta_true + sigma_true * torch.randn(n)

# Build model
sslr = SpikeSlabLinearRegression(n_features=p, pi=0.5, slab_scale=1)

# Train with SVI
optimizer = Adam({"lr": 1e-2})
svi = SVI(sslr.model, sslr.guide, optimizer, loss=Trace_ELBO())

num_steps = 3000
for step in range(num_steps):
    loss = svi.step(X, y)
    if (step + 1) % 500 == 0:
        print(f"step {step+1:4d} | loss/n = {loss / n:.4f}")

# ---------------------------
# Inspect variational posterior
# ---------------------------
alpha = torch.sigmoid(pyro.param("beta_alpha_logit")).detach()           # inclusion probs
mu = pyro.param("beta_mu").detach()                                      # slab means
beta_post_mean = alpha * mu                                              # E_q[gamma*z] = E[gamma]E[z]

print("\nPosterior inclusion probs alpha:")
print(alpha)

print("\nPosterior mean coefficients E[beta_j] ≈ alpha_j * mu_j:")
print(beta_post_mean)

print("\nTop active features by alpha:")
top_idx = torch.argsort(alpha, descending=True)[:10]
for j in top_idx.tolist():
    print(f"  j={j:2d}  alpha={alpha[j]:.3f}  mu={mu[j]:+.3f}  E[beta]={beta_post_mean[j]:+.3f}")

# ---------------------------
# Posterior predictive
# ---------------------------
predictive = Predictive(
    sslr.model,
    guide=sslr.guide,
    num_samples=500,
    return_sites=["obs", "intercept", "sigma", "beta_gamma", "beta_z"]
)

post = predictive(X[:5])  # shapes: [num_samples, ...]
y_samples = post["obs"]   # [500, 5]

pred_mean = y_samples.mean(0)
pred_std = y_samples.std(0)

print("\nPred mean (first 5):", pred_mean)
print("Pred std  (first 5):", pred_std)

step  500 | loss/n = 2.4533
step 1000 | loss/n = 2.6829
step 1500 | loss/n = 2.6615
step 2000 | loss/n = 2.8206
step 2500 | loss/n = 2.0117
step 3000 | loss/n = 2.8150

Posterior inclusion probs alpha:
tensor([0.6299, 0.4568, 0.2478, 0.2005, 0.2470, 0.3122, 0.4527, 0.3225, 0.1705,
        0.1331, 0.2018, 0.3327, 0.1859, 0.2787, 0.1765, 0.1393, 0.1197, 0.2545,
        0.0909, 0.2772])

Posterior mean coefficients E[beta_j] ≈ alpha_j * mu_j:
tensor([ 1.5370e+00, -7.7120e-01,  2.8722e-01, -3.7073e-03, -4.9989e-03,
         1.9551e-05,  2.9871e-02,  1.9388e-02,  3.2102e-02, -2.4165e-02,
        -6.0332e-03,  8.5560e-03, -3.6804e-03, -2.4596e-02,  3.4254e-03,
        -5.3695e-03,  1.9941e-02, -7.4095e-03, -2.5141e-03,  2.1858e-02])

Top active features by alpha:
  j= 0  alpha=0.630  mu=+2.440  E[beta]=+1.537
  j= 1  alpha=0.457  mu=-1.688  E[beta]=-0.771
  j= 6  alpha=0.453  mu=+0.066  E[beta]=+0.030
  j=11  alpha=0.333  mu=+0.026  E[beta]=+0.009
  j= 7  alpha=0.323  mu=+0.060  E[beta]=+0.0

In [3]:
post

{'beta_gamma': tensor([[[1., 0., 0.,  ..., 0., 0., 1.]],
 
         [[0., 0., 1.,  ..., 0., 0., 1.]],
 
         [[1., 1., 0.,  ..., 0., 0., 0.]],
 
         ...,
 
         [[1., 0., 0.,  ..., 1., 0., 1.]],
 
         [[1., 0., 1.,  ..., 0., 0., 1.]],
 
         [[1., 1., 0.,  ..., 0., 0., 1.]]]),
 'beta_z': tensor([[[ 2.3820, -1.6594,  0.9319,  ..., -0.1675, -0.0175,  0.0448]],
 
         [[ 2.2936, -1.6515,  1.2879,  ..., -0.1786, -0.0645,  0.1398]],
 
         [[ 2.4340, -1.6092,  1.1705,  ...,  0.0157,  0.0489, -0.1021]],
 
         ...,
 
         [[ 2.3866, -1.9078,  1.0909,  ..., -0.2459, -0.1102, -0.1088]],
 
         [[ 2.3387, -1.6998,  1.3143,  ..., -0.3663, -0.0819,  0.2218]],
 
         [[ 2.3568, -1.4943,  1.4201,  ..., -0.0441, -0.0523,  0.1109]]]),
 'intercept': tensor([[0.7394],
         [0.5641],
         [0.5952],
         [0.4597],
         [0.6827],
         [0.6317],
         [0.4838],
         [0.5397],
         [0.5105],
         [0.5382],
         [0.6431],
  